# D3 · M3.1 — Agent Fundamentals

**Worked side by side with the Concept HTML page, not read start to finish.** Read a short
concept section on the page, run the matching task below, then go back to the page for the next
concept — five round trips, not one long lab.

**THE SITUATION:** Day 2 built fixed chains and a full retrieval pipeline (M2.3/M2.4) where every
step was decided in advance, by you. Today the model decides its own steps. You'll build an agent
that picks between two tools — a calculator and a search over your own Day 2 Meridian knowledge
base — and watch it plan across both when one question needs them both.

**This notebook is fully working code — nothing to write.** Run each cell in order, read what
it prints, and follow the concept page's lab-marker cues to know when to come back here.

Concept page: `Day3/concept/D3_M3.1_Agent_Fundamentals_Concept.html`


## Setup

Loads your API key and the Day 2 Meridian article data this lab reuses. Run this once before
anything else.


In [ ]:
# Run once. "Requirement already satisfied" is a good outcome.
%pip install -q python-dotenv langchain langchain-openai


In [ ]:
import json
import time
from pathlib import Path

from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_openai import OpenAIEmbeddings

# Loads OPENAI_API_KEY from the .env file sitting next to this notebook. Same walk-up search
# every Day 2 notebook uses -- one key file for the whole repo, wherever you open this from.
def find_env():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / ".env").is_file():
            return candidate / ".env"
        legacy = candidate / "Day1" / "labs" / "starter" / ".env"
        if legacy.is_file():
            return legacy
    return None


_env = find_env()
if _env:
    load_dotenv(_env)
    print(f"Loaded API key from: {_env}")
else:
    print("WARNING: no .env file found -- see SETUP.md Step 2.")

import os
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set. Set it before continuing -- never hard-code a key in this notebook."
)


# Same walk-up idea, but for a data file instead of a .env -- avoids the hardcoded
# parents[N] bug earlier Day 2 notebooks had (see DELIVERY_PREREQS.md). Walks up from
# wherever this notebook is opened until it finds student_repo/Day2/data/... beneath it.
def find_repo_file(relative_path):
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        target = candidate / relative_path
        if target.is_file():
            return target
        target = candidate / "student_repo" / relative_path
        if target.is_file():
            return target
    raise FileNotFoundError(f"Could not find {relative_path} above {here}")


ARTICLES_PATH = find_repo_file(Path("Day2") / "data" / "D2_M2.3_meridian_articles.json")
with open(ARTICLES_PATH, "r", encoding="utf-8") as f:
    MERIDIAN_ARTICLES = json.load(f)
print(f"Loaded {len(MERIDIAN_ARTICLES)} Meridian articles from: {ARTICLES_PATH}")

model = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)


## Foundation, on the page — then come back here for Task 1

Read **What's An Agent, Really?** on the concept page (chatbot vs. chain vs. agent), then run the
cell below for Concept A.


## Concept A, on the page — then come back here for Task 1

Read **Concept A — Think → Act → Observe → Think Again** on the concept page and click through
the ReAct trace, then run the cell below — it builds and runs the same single-tool agent for real.


In [ ]:
# TASK 1 -- a single-tool agent. Nothing to edit here -- just run it and read the trace.
# The calculator tool: the model only ever sees the name and this one-line docstring below --
# never the code inside. Routing is the model matching a question against that description.
@tool
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, e.g. '12 * 4' or '2500 / 20'."""
    return str(eval(expression, {"__builtins__": {}}, {}))


single_tool_agent = create_agent(model, tools=[calculator])


def print_trace(result, label):
    """Prints every message in an agent run as a readable Thought/Action/Observation trace.
    This is today's actual "reasoning trace" deliverable -- not a black box, an ordered list
    of what the model decided and what came back from each tool it called."""
    print("=" * 72)
    print(label)
    print("=" * 72)
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "HumanMessage":
            print(f"  Question   : {msg.content}")
        elif role == "AIMessage" and getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                print(f"  Action     : {tc['name']}({tc['args']})")
        elif role == "ToolMessage":
            print(f"  Observation: {msg.content}")
        elif role == "AIMessage" and msg.content:
            print(f"  Final answer: {msg.content}")
    print()


task1_result = single_tool_agent.invoke(
    {"messages": [{"role": "user", "content": "What is 2500 divided by 20?"}]}
)
print_trace(task1_result, "TASK 1 -- single-tool agent (calculator only)")


**Notice:** the trace prints exactly what the concept page's click-through showed you --
Thought (implicit in the model choosing to call a tool), Action, Observation, then a final answer.
Now go back to the concept page for **Concept B**.


## Concept B, on the page — then come back here for Task 2

Read **Concept B — One Description, One Decision**, click all three demo questions, then run the
cell below — it adds a second real tool (search over your own Day 2 Meridian articles) and re-asks
a policy question and a math question.


In [ ]:
# TASK 2 -- add a second tool: search over the same 40 Meridian articles from Day 2's
# M2.3/M2.4 lab. Simple in-memory semantic search (like M1.4's hand-rolled version) --
# this module is about the AGENT deciding when to search, not about retrieval quality,
# which Day 2 already covered in depth.
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
ARTICLE_TEXTS = [f"{a['title']}. {a['text']}" for a in MERIDIAN_ARTICLES]
ARTICLE_VECTORS = embeddings_model.embed_documents(ARTICLE_TEXTS)
print(f"Embedded {len(ARTICLE_VECTORS)} articles for search_meridian_kb to use.")


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


@tool
def search_meridian_kb(query: str) -> str:
    """Search Meridian Bank's internal knowledge base for a policy, fee, or procedure answer.
    Use this for any question about account rules, limits, balances, or banking procedures."""
    query_vector = embeddings_model.embed_query(query)
    scored = [
        (cosine_similarity(query_vector, vec), article)
        for vec, article in zip(ARTICLE_VECTORS, MERIDIAN_ARTICLES)
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    best_score, best_article = scored[0]
    return f"[{best_article['id']}] {best_article['title']}. {best_article['text']}"


two_tool_agent = create_agent(model, tools=[calculator, search_meridian_kb])

task2_policy = two_tool_agent.invoke(
    {"messages": [{"role": "user",
                   "content": "What's the minimum balance for a metro savings account?"}]}
)
print_trace(task2_policy, "TASK 2a -- two-tool agent, policy question")

task2_math = two_tool_agent.invoke(
    {"messages": [{"role": "user", "content": "What is 15% of 40000?"}]}
)
print_trace(task2_math, "TASK 2b -- two-tool agent, math question")


**Notice:** the SAME agent, with both tools available the whole time, called a different
tool for each question -- because the descriptions matched differently, not because you told it
which one to use. Now go back to the concept page for **Concept C**.


## Concept C, on the page — then come back here for Task 3

Read **Concept C — When One Tool Isn't Enough**, click through the 5-step planning trace, then
run the cell below — the real UPI-limit question, forcing both tools in sequence.


In [ ]:
# TASK 3 -- a question that genuinely needs BOTH tools, in an order the agent has to work
# out for itself: look up the UPI limit first, THEN add up the three payments to compare.
# This is the module's core "reasoning trace" deliverable -- print it in full.
task3_question = (
    "A customer wants to send 40000, 35000 and 20000 rupees via UPI today. "
    "Will they breach the daily UPI limit, and by how much room is left or exceeded?"
)
task3_result = two_tool_agent.invoke(
    {"messages": [{"role": "user", "content": task3_question}]}
)
print_trace(task3_result, "TASK 3 -- multi-step planning (search THEN calculate)")

# A quick structural check, not a full grade: did the agent actually call both tools,
# in the order the question requires (search before calculate)?
tool_call_order = []
for msg in task3_result["messages"]:
    if msg.__class__.__name__ == "AIMessage" and getattr(msg, "tool_calls", None):
        tool_call_order.extend(tc["name"] for tc in msg.tool_calls)

print(f"Tools called, in order: {tool_call_order}")
if tool_call_order == ["search_meridian_kb", "calculator"]:
    print("Gold-tier routing: searched for the limit BEFORE calculating against it.")
else:
    print("Different order than expected -- read the trace above to see what happened instead.")


**Notice:** the printed order of tool calls above -- that ordering is the agent planning,
not you. Now go back to the concept page for **Concept D**.


## Concept D, on the page — then come back here for Tasks 4 and 5

Read **Concept D — The Most Important Slide Today: When NOT To Use One**, toggle the agent-vs-
plain-function comparison, then run the cells below — the real timing comparison, plus saving your
results.


In [ ]:
# TASK 4 -- agent vs. plain function, timed, on a FIXED, deterministic task (no judgment
# call needed). This is the concept page's Concept D toggle, run for real on your own machine.
fixed_expression = "12 * 4"

t0 = time.perf_counter()
agent_timed_result = single_tool_agent.invoke(
    {"messages": [{"role": "user", "content": f"What is {fixed_expression}?"}]}
)
agent_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
direct_result = calculator.func(fixed_expression)
direct_ms = (time.perf_counter() - t0) * 1000

print("=" * 72)
print("TASK 4 -- SAME TASK, THROUGH AN AGENT VS. A PLAIN FUNCTION CALL")
print("=" * 72)
print(f"Through the agent   : {agent_ms:8.1f} ms  (one LLM round trip to decide, then call the tool)")
print(f"Plain Python function: {direct_ms:8.2f} ms  (no LLM call -- there was nothing to decide)")
print(f"\nThe agent took roughly {agent_ms / max(direct_ms, 0.001):,.0f}x longer for an identical "
      "answer, because there was no real decision here for it to make. This is Concept D's point, "
      "made with real numbers from your own run: match the tool to the job.")


In [ ]:
# TASK 5 -- save your results, and an optional LangSmith tracing add-on.
#
# OPTIONAL / STRETCH: LangSmith gives you the same Thought/Action/Observation trace you've been
# printing by hand above, but as a shareable, clickable timeline -- no code change needed beyond
# two environment variables. Per this programme's tech-stack default, this is an
# instructor-demo-only feature unless your VM's LangSmith connectivity has been confirmed --
# uncomment and set these only if your trainer has told you it's available:
#
# import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "<your LangSmith API key>"
# (Re-run Tasks 1-3 above after setting these -- every agent.invoke() call is traced automatically.)

key_takeaway = """
KEY TAKEAWAY: An agent is autonomy + tools + a feedback loop, together. Task 1 showed the
loop with one tool. Task 2 showed the SAME agent routing correctly between two tools, based
only on their descriptions. Task 3 showed it planning a two-step path -- search, then
calculate -- that nobody hard-coded. Task 4 showed why that power isn't free: for a fixed,
deterministic task, a plain function beat the agent by a wide margin, with an identical answer.
"""
print(key_takeaway)

results = {
    "task1_single_tool_final_answer": task1_result["messages"][-1].content,
    "task2_policy_tool_called": [
        tc["name"] for msg in task2_policy["messages"]
        if msg.__class__.__name__ == "AIMessage" and getattr(msg, "tool_calls", None)
        for tc in msg.tool_calls
    ],
    "task2_math_tool_called": [
        tc["name"] for msg in task2_math["messages"]
        if msg.__class__.__name__ == "AIMessage" and getattr(msg, "tool_calls", None)
        for tc in msg.tool_calls
    ],
    "task3_tool_call_order": tool_call_order,
    "task3_final_answer": task3_result["messages"][-1].content,
    "task4_agent_ms": agent_ms,
    "task4_direct_ms": direct_ms,
    "key_takeaway": key_takeaway,
}

with open("m3_1_agent_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved m3_1_agent_results.json --", len(json.dumps(results)), "bytes")
